In [ ]:
!pip install -q langchain langchain-groq gspread

In [ ]:
import os
from google.colab import userdata, auth
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.2, max_retries=3)

auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)
sh = gc.create("Sweet Crumbs Orders")      # run once; next time use gc.open("Sweet Crumbs Orders")
ws = sh.sheet1
ws.append_row(["name", "item", "qty", "date", "eggless", "total"])
print("Sheet ready:", sh.url)

Sheet ready: https://docs.google.com/spreadsheets/d/1DMIR3L3tXV45ntkwf6x7-KvhIXF0778aqtFnwWN-sYo


In [29]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool

MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}
HOURS = {"monday": "9 am to 8 pm", "tuesday": "9 am to 8 pm", "wednesday": "9 am to 8 pm",
         "thursday": "9 am to 8 pm", "friday": "9 am to 8 pm", "saturday": "9 am to 8 pm",
         "sunday": "10 am to 6 pm"}


class Order(BaseModel):
    """A cake order taken from a customer's chat message."""
    name: str = Field(description="Customer's first name")
    item: Literal["plum cake", "chocolate cake", "cupcake"] = Field(description="Menu item")
    qty: int = Field(description="How many", ge=1)
    date: str = Field(description="Pickup date as the customer said it")
    eggless: bool = Field(description="True if the customer asked for eggless")

@tool
def get_price(item: str, qty: int = 1) -> str:
    """Get the total price in rupees for a quantity of one menu item.

    Args:
        item: Menu item, e.g. "plum cake", "chocolate cake" or "cupcake"
        qty: How many the customer wants
    """
    price = MENU.get(item.lower().strip())
    if price is None:
        return f"'{item}' is not on the menu. We have: {', '.join(MENU)}."
    return f"{qty} x {item} = Rs {price * qty}"

@tool
def is_open(day: str) -> str:
    """Check the bakery's opening hours for a day of the week, e.g. "sunday"."""
    hours = HOURS.get(day.lower().strip())
    return f"Open {hours} on {day}." if hours else f"'{day}' is not a day of the week."

@tool
def log_order(name: str, item: str, qty: int, date: str, eggless: bool) -> str:
    """Save a confirmed cake order to the Orders sheet.
    Use only when the customer has given their name, item, quantity and pickup date."""
    try:
        order = Order(name=name, item=item.lower().strip(), qty=qty, date=date, eggless=eggless)
    except Exception as e:
        return f"Could not save the order: {e}. Ask the customer to correct it."
    total = MENU[order.item] * order.qty
    ws.append_row([order.name, order.item, order.qty, order.date,
                   "yes" if order.eggless else "no", total])
    return f"Saved: {order.qty} x {order.item} for {order.name} on {order.date}, Rs {total}."

@tool
def tool1(item: str) -> str:
    """Does stuff."""
    return f"{item} your query is not clear please try again"

In [30]:
from langchain.agents import create_agent

SYSTEM = """ROLE:   You are the friendly order assistant for Sweet Crumbs Home Bakery.
FACTS:  Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Always use get_price for prices and is_open for timings; never guess them.
        Before log_order, make sure you have name, item, quantity and pickup date; ask if missing.
        After saving, read the order back with the total.
FORMAT: Under 30 words, friendly, English or Telugu matching the customer."""

agent = create_agent(model=llm, tools=[get_price, is_open, log_order], system_prompt=SYSTEM)

history = []
def chat(text: str) -> str:
    global history
    try:
        result = agent.invoke({"messages": history + [{"role": "user", "content": text}]})
        history = result["messages"]                  # keep the whole run as memory
        return history[-1].content
    except Exception as e:
        print("Agent error:", e)
        return "Sorry, I'm a little busy. Please try again in a minute."

print("WELCOME TO Sweet Crumbs Barkery :\n")
while True :
  s = input("User: ", )
  if s in {"end","bye","stop","quit"}:
    print("Thank you, have a nice day")
    break
  print(f"Bot : {chat(s)}\n")

   # check your sheet!

WELCOME TO Sweet Crumbs Barkery :

User: hi
Bot : Hi! 😊 What would you like to order (cake or cupcakes), and when would you like to pick it up?Sure! What would you like to order (cake or cupcakes) and your preferred pickup date?

User: hello
Bot : Hello! 😊 What would you like to order, how many, your name, and the pickup date?

User: i am harsha list the items
Bot : Hi Harsha! 🌟 Our menu: plum cake, chocolate cake, carrot cake, red‑velvet cake, vanilla cupcake, chocolate cupcake, strawberry cupcake. Which would you like?

User: i want 3 cupcakes , plum cake
Bot : Great! Which cupcake flavor and how many plum cakes? Also, what pickup date would you like?

User: carrot , all eggles and for friday
Bot : Harsha, could you tell me which cupcake flavor you’d like (vanilla, chocolate or strawberry) and confirm 1 plum cake? Then I’ll lock in your egg‑less carrot cake for Friday.

User: yes
Bot : Harsha, please confirm:  

- Cupcake flavor (vanilla, chocolate or strawberry) – 3 pieces  
- Plum 